# CutTrack

Ett verktyg för att logga vikt, kalorier, protein, steg och träning under en deff, och få regelbaserade råd om vikten och kosten rör sig åt rätt håll.

Notebooken byggs upp del för del. I det här steget finns:

- `DailyLog`, klassen som representerar en enskild dags logg
- `User`, basklassen med användarens grunddata och metoderna som räknar på loggarna
- `log_today`, funktionen som frågar användaren om dagens värden och lägger till en logg

`CutProfile`, barnklassen som ärver från `User` och lägger till mål och kaloriberäkningar, byggs i nästa steg.

In [55]:
from datetime import datetime

## DailyLog

Representerar en dags logg. Kontrollerar i `__init__` att vikt och kalorier är rimliga tal. Är de inte det kastas ett `ValueError` med ett förklarande meddelande, som fångas längre ner i `log_today`.

Gränserna (0 till 300 kg, 0 till 10 000 kcal) är satta för att fånga uppenbara skrivfel, som ett extra 0 eller ett minustecken, inte för att vara medicinskt exakta.

`trained` är en bool, `True` om personen tränade den dagen. Under en deff är det träningsfrekvensen som håller muskelmassan uppe, inte hur många minuter passet varade. Ett långt pass med mycket vila ger inte mer stimulans än ett kort och fokuserat. Frekvens går dessutom att bedöma mot en tydlig regel, antal dagar per vecka, medan minuter kräver ett godtyckligt tröskelvärde.

`waist` har standardvärdet `None`, eftersom midjemått är en valfri mätning. Att `None` inte är samma sak som 0 spelar roll längre fram, när snitt ska räknas och skilja på utebliven mätning och ett mätvärde på 0 cm.

In [56]:
class DailyLog:
    """En daglig logg med vikt, kalorier och annan data för ett datum."""

    def __init__(self, date, weight, calories, protein, steps, trained, waist=None):
        if weight <= 0 or weight > 300:
            raise ValueError("Vikten måste vara ett rimligt tal i kilogram, till exempel 82.5.")
        if calories < 0 or calories > 10000:
            raise ValueError("Kalorierna måste vara ett rimligt tal, till exempel 2200.")

        self.date = date
        self.weight = weight
        self.calories = calories
        self.protein = protein
        self.steps = steps
        self.trained = trained
        self.waist = waist

## User

Basklassen för en användare. Innehåller personens grunddata och listan med loggar, plus metoderna som räknar på loggarna.

`logs` börjar som en tom lista. Varje logg som läggs till är ett `DailyLog`-objekt.

Metoderna:

- `add_log`, lägger till en logg, eller byter ut den befintliga om det redan finns en logg för samma datum
- `get_logs(days)`, plockar ut loggarna från de senaste `days` kalenderdagarna
- `average_weight(days)`, medelvikt över perioden
- `weight_change(days)`, hur mycket vikten ändrats över perioden
- `training_days(days)`, antal dagar med träning under perioden

In [57]:
class User:
    """Basklass för en användare av CutTrack."""

    def __init__(self, name, height_cm, age, sex, activity_level, start_weight, created_date):
        self.name = name
        self.height_cm = height_cm
        self.age = age
        self.sex = sex
        self.activity_level = activity_level
        self.start_weight = start_weight
        self.created_date = created_date
        self.logs = []

    def add_log(self, new_log):
        for i in range(len(self.logs)):
            if self.logs[i].date == new_log.date:
                self.logs[i] = new_log
                print(f"Loggen för {new_log.date} uppdaterades.")
                return
        self.logs.append(new_log)
        print(f"Loggen för {new_log.date} lades till.")

    def get_logs(self, days):
        """Returnerar loggarna från de senaste days kalenderdagarna."""
        if len(self.logs) == 0:
            return []

        latest_date = None
        for log in self.logs:
            log_date = datetime.strptime(log.date, "%Y-%m-%d")
            if latest_date is None or log_date > latest_date:
                latest_date = log_date

        selected_logs = []
        for log in self.logs:
            log_date = datetime.strptime(log.date, "%Y-%m-%d")
            days_ago = (latest_date - log_date).days
            if days_ago < days:
                selected_logs.append(log)

        return selected_logs

    def average_weight(self, days):
        """Medelvikt över perioden. Returnerar None om det inte finns några loggar."""
        selected_logs = self.get_logs(days)

        if len(selected_logs) == 0:
            return None

        total_weight = 0
        for log in selected_logs:
            total_weight = total_weight + log.weight

        return total_weight / len(selected_logs)

    def average_protein(self, days):
        """Medelprotein i gram över perioden. Returnerar None om det inte finns några loggar."""
        selected_logs = self.get_logs(days)

        if len(selected_logs) == 0:
            return None

        total_protein = 0
        for log in selected_logs:
            total_protein = total_protein + log.protein

        return total_protein / len(selected_logs)

    def average_steps(self, days):
        """Medelantal steg över perioden. Returnerar None om det inte finns några loggar."""
        selected_logs = self.get_logs(days)

        if len(selected_logs) == 0:
            return None

        total_steps = 0
        for log in selected_logs:
            total_steps = total_steps + log.steps

        return total_steps / len(selected_logs)

    def weight_change(self, days):
        """Viktförändring i kg över perioden. Negativt tal betyder att vikten gått ner.
        Returnerar None om det finns färre än två loggar."""
        selected_logs = self.get_logs(days)

        if len(selected_logs) < 2:
            return None

        first_log = selected_logs[0]
        last_log = selected_logs[0]
        for log in selected_logs:
            if log.date < first_log.date:
                first_log = log
            if log.date > last_log.date:
                last_log = log

        return last_log.weight - first_log.weight

    def training_days(self, days):
        """Antal dagar med träning under perioden."""
        selected_logs = self.get_logs(days)

        total_days = 0
        for log in selected_logs:
            if log.trained:
                total_days = total_days + 1

        return total_days

### Hur get_logs räknar kalenderdagar

Enligt teknisk_plan.md ska fönstret räknas på kalenderdagar, inte på antal loggar. Sju loggar utspridda över en månad är inte ett veckosnitt.

Metoden gör tre saker:

1. Loopar igenom alla loggar och hittar det senaste datumet. `datetime.strptime` gör om datumsträngen till ett datumobjekt som går att jämföra med `>`.
2. Loopar igenom alla loggar igen och räknar ut hur många dagar bakåt varje logg ligger, genom att dra det ena datumet från det andra och läsa `.days`.
3. Tar med loggen om den ligger inom fönstret.

`average_weight`, `weight_change` och `training_days` anropar alla `get_logs` först, så urvalsregeln finns på ett enda ställe. Ändras regeln behöver bara `get_logs` ändras.

`weight_change` sorterar inte listan. Den loopar igenom och hittar den tidigaste och den senaste loggen genom att jämföra datumsträngarna direkt. Det fungerar eftersom formatet ÅÅÅÅ-MM-DD sorteras rätt som text, vilket är skälet till att just det formatet valdes i teknisk_plan.md.

`average_weight` och `weight_change` returnerar `None` när underlaget är för tunt, inga loggar alls respektive färre än två. Det skiljer sig från att returnera 0, som skulle betyda att vikten inte ändrades. `training_days` returnerar däremot 0 när det inte finns några loggar, eftersom noll träningsdagar är ett korrekt svar och inte ett saknat värde.

## CutProfile, barnklass med arv

`CutProfile` ärver från `User`. Allt som `User` kan, kan `CutProfile` också: `add_log`, `get_logs`, `average_weight`, `weight_change` och `training_days` finns utan att skrivas om.

Det som läggs till är målen och kaloriberäkningarna, alltså det som är specifikt för någon som deffar. En framtida `BulkProfile` eller `MaintenanceProfile` skulle kunna ärva från samma `User` utan att röra den koden.

`super().__init__(...)` anropar förälderklassens `__init__`, som sätter namn, längd, ålder, kön, aktivitetsnivå, startvikt, datum och den tomma logglistan. Sedan sätter `CutProfile` sina egna attribut. Utan `super()` skulle inget av `User`-attributen finnas, och `self.logs` skulle inte existera.

`protein_goal_per_kg` och `step_goal` har standardvärden, så de kan utelämnas när en profil skapas. `calorie_goal` sätts till `None` från start eftersom den räknas ut först senare, av `suggest_calorie_goal`.

In [58]:
class CutProfile(User):
    """En användare som deffar. Ärver allt från User och lägger till mål och kaloriberäkningar."""

    def __init__(self, name, height_cm, age, sex, activity_level, start_weight, created_date,
                 goal_weight, target_rate_percent, protein_goal_per_kg=1.9, step_goal=8000,
                 training_goal_days=3):
        super().__init__(name, height_cm, age, sex, activity_level, start_weight, created_date)

        if goal_weight <= 0 or goal_weight >= start_weight:
            raise ValueError("Målvikten måste vara lägre än startvikten.")
        if target_rate_percent <= 0 or target_rate_percent > 1.0:
            raise ValueError("Takten måste ligga mellan 0 och 1,0 procent av kroppsvikten per vecka.")

        self.goal_weight = goal_weight
        self.target_rate_percent = target_rate_percent
        self.protein_goal_per_kg = protein_goal_per_kg
        self.step_goal = step_goal
        self.training_goal_days = training_goal_days
        self.calorie_goal = None

    def current_weight(self):
        """Senaste loggade vikten. Startvikten om inga loggar finns."""
        if len(self.logs) == 0:
            return self.start_weight

        latest_log = self.logs[0]
        for log in self.logs:
            if log.date > latest_log.date:
                latest_log = log

        return latest_log.weight

    def calculate_bmr(self):
        """Basalomsättning i kalorier per dygn, enligt Mifflin-St Jeor."""
        weight = self.current_weight()
        bmr = 10 * weight + 6.25 * self.height_cm - 5 * self.age

        if self.sex == "man":
            bmr = bmr + 5
        else:
            bmr = bmr - 161

        return bmr

    def calculate_tdee(self):
        """Totalt dagligt energibehov, basalomsättningen gånger aktivitetsfaktorn."""
        return self.calculate_bmr() * self.activity_level

    def protein_goal(self):
        """Proteinmål i gram per dag, räknat på målvikten."""
        return self.goal_weight * self.protein_goal_per_kg

    def suggest_calorie_goal(self):
        """Räknar ut ett dagligt kaloriförslag utifrån önskad takt.
        Går aldrig under golvet, som är det högsta av basalomsättningen
        och en fast gräns (1500 kcal för män, 1200 för kvinnor)."""
        tdee = self.calculate_tdee()
        bmr = self.calculate_bmr()
        weight = self.current_weight()

        weekly_loss_kg = weight * self.target_rate_percent / 100
        daily_deficit = weekly_loss_kg * 7700 / 7
        suggested_calories = tdee - daily_deficit

        if self.sex == "man":
            fixed_floor = 1500
        else:
            fixed_floor = 1200

        if bmr > fixed_floor:
            floor = bmr
        else:
            floor = fixed_floor

        if suggested_calories < floor:
            print("Kaloriförslaget hamnade under ditt golv och har justerats upp.")
            print(f"Golvet är {round(floor)} kcal, det högsta av din basalomsättning "
                  f"({round(bmr)}) och den fasta gränsen ({fixed_floor}).")
            suggested_calories = floor

            actual_deficit = tdee - floor
            if actual_deficit <= 0:
                print("Med ditt golv går det inte att skapa något underskott genom maten.")
                print("Vägen framåt är att röra dig mer, inte att äta mindre.")
            else:
                actual_weekly_loss = actual_deficit * 7 / 7700
                actual_rate = actual_weekly_loss / weight * 100
                print(f"Takten blir därför {round(actual_rate, 2)} procent per vecka "
                      f"i stället för {self.target_rate_percent}.")
                print("Vill du gå ner snabbare är vägen dit att röra dig mer, inte att äta mindre.")

        self.calorie_goal = round(suggested_calories)
        return self.calorie_goal

    def days_since_start(self):
        """Antal dagar sedan profilen skapades, räknat till senaste loggade dagen."""
        if len(self.logs) == 0:
            return 0

        latest_log = self.logs[0]
        for log in self.logs:
            if log.date > latest_log.date:
                latest_log = log

        start_date = datetime.strptime(self.created_date, "%Y-%m-%d")
        latest_date = datetime.strptime(latest_log.date, "%Y-%m-%d")
        return (latest_date - start_date).days + 1

    def waiting_message(self):
        """Text för dag 1 till 13, innan full analys är möjlig. Returnerar None från dag 14."""
        days = self.days_since_start()

        if days < 7:
            remaining = 7 - days
            return ("Vikten svänger flera hundra gram från dag till dag på grund av vätska och "
                    "maginnehåll, mer än en hel veckas verkliga fettförlust. Ett råd byggt på "
                    f"{days} dagars data vore bara brus. {remaining} dagar kvar tills ett "
                    "sjudagarssnitt visas.")

        if days < 14:
            remaining = 14 - days
            return ("Sjudagarssnittet är nu tillräckligt med data för att visas, men takten går "
                    f"inte att bedöma säkert än. {remaining} dagar kvar tills full analys är möjlig.")

        return None

    def check_goals(self):
        """Statusöversikt över vikt, protein, träningsfrekvens och steg, plus en sak att
        fokusera på. Prioritetsordning: vikt, protein, träningsfrekvens, steg."""
        waiting_text = self.waiting_message()
        if waiting_text is not None:
            print(waiting_text)
            return

        focus_area = None

        # Vikt: undre gränsen är personens eget mål, övre gränsen är ett fast säkerhetstak
        weekly_change = self.weight_change(7)
        avg_weight = self.average_weight(7)
        if weekly_change is not None and avg_weight is not None and avg_weight > 0:
            rate_percent = -weekly_change / avg_weight * 100
            if weekly_change > 0:
                print(f"Vikten: ökar med {round(weekly_change, 1)} kg den senaste veckan. "
                      "Underskottet räcker inte.")
                focus_area = "vikt"
            elif rate_percent > 1.0:
                print(f"Vikten: minskar med {round(rate_percent, 2)} procent per vecka, över "
                      "säkerhetsgränsen 1,0 procent. Risk för muskelförlust.")
                if focus_area is None:
                    focus_area = "vikt"
            elif rate_percent < self.target_rate_percent:
                print(f"Vikten: minskar med {round(rate_percent, 2)} procent per vecka, "
                      f"långsammare än ditt mål på {self.target_rate_percent} procent.")
                if focus_area is None:
                    focus_area = "vikt"
            else:
                print(f"Vikten: minskar med {round(rate_percent, 2)} procent per vecka, vid "
                      f"eller över ditt mål på {self.target_rate_percent} procent och inom "
                      "säkerhetsgränsen.")
        else:
            print("Vikten: för lite data för att bedöma takten.")

        # Protein
        avg_protein = self.average_protein(7)
        protein_target = self.protein_goal()
        if avg_protein is not None:
            if avg_protein < protein_target:
                print(f"Protein: snitt {round(avg_protein)} g mot mål {round(protein_target)} g. "
                      "Under målet.")
                if focus_area is None:
                    focus_area = "protein"
            else:
                print(f"Protein: snitt {round(avg_protein)} g mot mål {round(protein_target)} g. "
                      "Målet nås.")
        else:
            print("Protein: för lite data.")

        # Träningsfrekvens
        training_count = self.training_days(7)
        if training_count < self.training_goal_days:
            print(f"Träning: {training_count} av {self.training_goal_days} planerade dagar den "
                  "senaste veckan. Under målet.")
            if focus_area is None:
                focus_area = "träning"
        else:
            print(f"Träning: {training_count} av {self.training_goal_days} planerade dagar den "
                  "senaste veckan. Målet nås.")

        # Steg
        avg_steps = self.average_steps(7)
        if avg_steps is not None:
            if avg_steps < self.step_goal:
                print(f"Steg: snitt {round(avg_steps)} mot mål {self.step_goal}. Under målet.")
                if focus_area is None:
                    focus_area = "steg"
            else:
                print(f"Steg: snitt {round(avg_steps)} mot mål {self.step_goal}. Målet nås.")
        else:
            print("Steg: för lite data.")

        print()
        if focus_area is None:
            print("Allt ligger inom mål just nu. Fortsätt som du gör.")
        else:
            print(f"Fokusera på: {focus_area}.")

### Om beräkningarna

**Mifflin-St Jeor** räknar ut basalomsättningen, alltså hur många kalorier kroppen gör av med i vila. Formeln är densamma för båda könen så när som på sista termen, plus 5 för män och minus 161 för kvinnor. Därför står den gemensamma delen först och if-satsen justerar bara slutet.

**`current_weight`** finns för att BMR ska räknas på vad personen väger nu, inte på startvikten. Under en deff sjunker vikten, och därmed sjunker också förbrukningen. Räknas BMR på startvikten överskattas behovet mer och mer ju längre deffen pågår.

Den hittar senaste loggen genom att jämföra datumsträngar, precis som `weight_change`, och faller tillbaka på `start_weight` när det inte finns några loggar än.

**`calculate_tdee`** multiplicerar med aktivitetsfaktorn: 1,2 stillasittande, 1,375 lätt aktiv, 1,55 måttligt aktiv, 1,725 mycket aktiv, 1,9 extremt aktiv.

**`protein_goal`** räknas på målvikten och inte på nuvarande vikt. Målet ligger därmed fast under hela deffen i stället för att sjunka i takt med att vikten går ner, vilket vore fel: proteinbehovet finns för att skydda den muskelmassa som ska vara kvar när deffen är slut.

**Valideringen** i `__init__` fångar två saker som gör resten av beräkningarna meningslösa: en målvikt som är högre än startvikten, och en takt över 1,0 procent per vecka, där risken för muskelförlust blir för stor. Samma mönster som i `DailyLog`, felet kastas där objektet skapas.

### Test av CutProfile

Testcellen visar tre saker: att formeln stämmer mot handräknat facit, att de ärvda metoderna från `User` fungerar utan att vara omskrivna, och att `isinstance` bekräftar arvet.

In [46]:
profile = CutProfile("Niklas", 183, 39, "man", 1.55, 90.0, "2026-09-01",
                     goal_weight=82.0, target_rate_percent=0.7)

print("Basalomsättning:", round(profile.calculate_bmr(), 1))
print("Energibehov:", round(profile.calculate_tdee(), 1))
print("Proteinmål:", round(profile.protein_goal(), 1), "gram")
print("Aktuell vikt utan loggar:", profile.current_weight())

profile.add_log(DailyLog("2026-09-15", 89.0, 2200, 150, 8000, True))
profile.add_log(DailyLog("2026-09-16", 88.6, 2150, 160, 9000, False))

print("Aktuell vikt med loggar:", profile.current_weight())
print("Basalomsättning nu:", round(profile.calculate_bmr(), 1))
print("Medelvikt 7 dagar:", profile.average_weight(7))
print("Träningsdagar 7 dagar:", profile.training_days(7))
print("Är CutProfile också en User?", isinstance(profile, User))

Basalomsättning: 1853.8
Energibehov: 2873.3
Proteinmål: 155.8 gram
Aktuell vikt utan loggar: 90.0
Loggen för 2026-09-15 lades till.
Loggen för 2026-09-16 lades till.
Aktuell vikt med loggar: 88.6
Basalomsättning nu: 1839.8
Medelvikt 7 dagar: 88.8
Träningsdagar 7 dagar: 1
Är CutProfile också en User? True


In [48]:
# Valideringen ska stoppa orimliga profiler
try:
    CutProfile("Test", 180, 30, "man", 1.2, 80.0, "2026-09-01",
               goal_weight=85.0, target_rate_percent=0.5)
except ValueError as error:
    print("Fångat fel:", error)

try:
    CutProfile("Test", 180, 30, "man", 1.2, 80.0, "2026-09-01",
               goal_weight=75.0, target_rate_percent=1.5)
except ValueError as error:
    print("Fångat fel:", error)

Fångat fel: Målvikten måste vara lägre än startvikten.
Fångat fel: Takten måste ligga mellan 0 och 1,0 procent av kroppsvikten per vecka.


### suggest_calorie_goal

Metoden räknar ut hur många kalorier personen bör äta per dag, och är den enda i projektet som kan vägra göra som användaren bett om.

**Steg för steg:**

1. Räkna ut hur många kilo i veckan den önskade takten motsvarar: vikten gånger procenttalet delat med 100.
2. Gör om det till ett dagligt underskott med tumregeln 7700 kcal per kilo kroppsfett, delat på veckans sju dagar.
3. Dra underskottet från energibehovet.
4. Kontrollera mot golvet, och justera upp om förslaget hamnade under.

**Golvet** är det högsta av två värden: den fasta gränsen (1500 kcal för män, 1200 för kvinnor) och personens egen basalomsättning. Är basalomsättningen 1780 blir golvet 1780, inte 1500. Fördelen med det rörliga golvet är att det skalar med personen, vilket ett fast tal inte gör.

**Programmet justerar inte tyst.** Slår golvet i skrivs det ut vad som hände, vilket golv som gällde, och vilken takt som faktiskt blir möjlig. Tyst justering skulle få användaren att tro att den snabba takten fortfarande gäller.

**En konsekvens värd att känna till:** en stillasittande person har aktivitetsfaktor 1,2, alltså är energibehovet bara 20 procent över basalomsättningen. Största möjliga underskott blir därmed litet, och takten hamnar ofta runt 0,4 procent per vecka oavsett vad personen önskat. Det är inte en bugg. Slutsatsen programmet drar är att vägen till snabbare viktnedgång går via mer aktivitet, inte via mindre mat, och det säger det uttryckligen.

**Om 7700-regeln:** siffran kommer från en artikel från 1958 och antar ett konstant energiunderskott, vilket inte stämmer. När vikten sjunker sjunker även förbrukningen, så samma underskott ger mindre effekt över tid. Regeln överskattar alltså den faktiska viktnedgången. Den används här som startpunkt just för att den är enkel och går att förklara, och tanken är att siffran justeras mot verkligt utfall efter ett par veckor. Det är ett medvetet val av en enkel modell med kända begränsningar.

### Test av suggest_calorie_goal

Tre fall: ett där golvet inte slår i, ett där det gör det, och ett där energibehovet är så lågt att inget underskott alls går att skapa genom maten.

In [49]:
print("Fall 1, golvet slår inte i")
a = CutProfile("Niklas", 183, 39, "man", 1.55, 90.0, "2026-09-01",
               goal_weight=82.0, target_rate_percent=0.7)
print("Energibehov:", round(a.calculate_tdee()), "Basalomsättning:", round(a.calculate_bmr()))
print("Kaloriförslag:", a.suggest_calorie_goal())
print()

print("Fall 2, golvet slår i")
b = CutProfile("Testperson", 155, 30, "kvinna", 1.2, 55.0, "2026-09-01",
               goal_weight=50.0, target_rate_percent=1.0)
print("Energibehov:", round(b.calculate_tdee()), "Basalomsättning:", round(b.calculate_bmr()))
print("Kaloriförslag:", b.suggest_calorie_goal())
print()

print("Fall 3, inget underskott möjligt genom maten")
c = CutProfile("Testperson", 150, 65, "kvinna", 1.2, 48.0, "2026-09-01",
               goal_weight=45.0, target_rate_percent=1.0)
print("Energibehov:", round(c.calculate_tdee()), "Basalomsättning:", round(c.calculate_bmr()))
print("Kaloriförslag:", c.suggest_calorie_goal())

Fall 1, golvet slår inte i
Energibehov: 2873 Basalomsättning: 1854
Kaloriförslag: 2180

Fall 2, golvet slår i
Energibehov: 1449 Basalomsättning: 1208
Kaloriförslaget hamnade under ditt golv och har justerats upp.
Golvet är 1208 kcal, det högsta av din basalomsättning (1208) och den fasta gränsen (1200).
Takten blir därför 0.4 procent per vecka i stället för 1.0.
Vill du gå ner snabbare är vägen dit att röra dig mer, inte att äta mindre.
Kaloriförslag: 1208

Fall 3, inget underskott möjligt genom maten
Energibehov: 1118 Basalomsättning: 932
Kaloriförslaget hamnade under ditt golv och har justerats upp.
Golvet är 1200 kcal, det högsta av din basalomsättning (932) och den fasta gränsen (1200).
Med ditt golv går det inte att skapa något underskott genom maten.
Vägen framåt är att röra dig mer, inte att äta mindre.
Kaloriförslag: 1200


### days_since_start och waiting_message

Dagliga viktsvängningar från vätska och maginnehåll ligger ofta på flera hundra gram, mer än en hel veckas verkliga fettförlust. Ett råd byggt på tre dagars data är brus, inte information.

`days_since_start` räknar från `created_date` till den senast loggade dagen, inte till dagens riktiga datum. Det håller samma princip som `get_logs`: har du inte loggat på några dagar ska analysen ändå utgå från din senaste aktiva period.

`waiting_message` returnerar text dag 1 till 13, och `None` från dag 14. `check_goals` kollar detta först, och avbryter tidigt om det finns text att visa i stället för en analys.

### check_goals

Ger en statusrad för vikt, protein, träningsfrekvens och steg, i den ordningen, och lyfter sedan fram en enda sak att fokusera på.

**Vikten** bedöms mot två olika gränser med olika syften. Den undre gränsen är personens eget `target_rate_percent`, vad de själva bett om. Den övre gränsen är ett fast säkerhetstak på 1,0 procent per vecka, oavsett vad personen satt som mål. Skälet till skillnaden: hur snabbt någon vill gå ner är en fråga om preferens, men risken för muskelförlust vid för snabb nedgång är en fråga om säkerhet, och säkerhet ska inte gå att ställa in bort. Sätter någon sitt mål till 0,3 procent och landar på 0,3, är det precis vad de bad om, inte för långsamt. Landar de på 1,3, flaggas det, även om de själva satt målet till 1,0.

**Protein, träning och steg** kräver bara ett snitt, ingen trend, och analyseras därför oavsett hur många dagar som gått, till skillnad från vikten.

**`focus_area`** sätts bara av det första området som inte når sitt mål, tack vare `if focus_area is None:` i varje gren. Vikt kollas först i koden, så ett viktproblem vinner alltid över ett proteinproblem, oavsett i vilken ordning de faktiskt upptäcks. Är allt inom mål förblir `focus_area` `None`, och en positiv sammanfattning skrivs i stället, byggd på att inget slog till, inte på en generell fras.

### Test av check_goals

Fyra scenarier: tidig väntetext, allt inom mål, vikten går fel håll, och ett lågt personligt mål som ändå nås snabbare men säkert och därför inte flaggas.

In [59]:
from datetime import timedelta

def add_days_of_logs(profile, start, num_days, weight_start, daily_change,
                      calories, protein, steps, trained_pattern):
    """Hjälpfunktion bara för testning, lägger till num_days loggar i rad."""
    start_date = datetime.strptime(start, "%Y-%m-%d")
    for i in range(num_days):
        date_text = (start_date + timedelta(days=i)).strftime("%Y-%m-%d")
        weight = round(weight_start + daily_change * i, 1)
        trained = trained_pattern[i % len(trained_pattern)]
        profile.add_log(DailyLog(date_text, weight, calories, protein, steps, trained))

In [60]:
print("Dag 3, väntetext:")
early = CutProfile("Test", 183, 39, "man", 1.55, 90.0, "2026-09-01", 82.0, 0.7)
add_days_of_logs(early, "2026-09-01", 3, 90.0, -0.05, 2200, 150, 8000, [True, False, True])
early.check_goals()
print()

print("Dag 20, allt inom mål:")
good = CutProfile("Test", 183, 39, "man", 1.55, 90.0, "2026-09-01", 82.0, 0.7)
add_days_of_logs(good, "2026-09-01", 20, 90.0, -0.10, 2000, 175, 9000, [True, True, False, True])
good.check_goals()
print()

print("Dag 20, vikten ökar:")
wrong_way = CutProfile("Test", 183, 39, "man", 1.55, 90.0, "2026-09-01", 82.0, 0.7)
add_days_of_logs(wrong_way, "2026-09-01", 20, 90.0, 0.05, 2600, 175, 9000, [True, True, False, True])
wrong_way.check_goals()
print()

print("Dag 20, lågt personligt mål, snabbare men säkert, ska inte flaggas:")
low_goal = CutProfile("Test", 183, 39, "man", 1.55, 90.0, "2026-09-01", 88.0, 0.3)
add_days_of_logs(low_goal, "2026-09-01", 20, 90.0, -0.10, 2000, 175, 9000, [True, True, False, True])
low_goal.check_goals()

Dag 3, väntetext:
Loggen för 2026-09-01 lades till.
Loggen för 2026-09-02 lades till.
Loggen för 2026-09-03 lades till.
Vikten svänger flera hundra gram från dag till dag på grund av vätska och maginnehåll, mer än en hel veckas verkliga fettförlust. Ett råd byggt på 3 dagars data vore bara brus. 4 dagar kvar tills ett sjudagarssnitt visas.

Dag 20, allt inom mål:
Loggen för 2026-09-01 lades till.
Loggen för 2026-09-02 lades till.
Loggen för 2026-09-03 lades till.
Loggen för 2026-09-04 lades till.
Loggen för 2026-09-05 lades till.
Loggen för 2026-09-06 lades till.
Loggen för 2026-09-07 lades till.
Loggen för 2026-09-08 lades till.
Loggen för 2026-09-09 lades till.
Loggen för 2026-09-10 lades till.
Loggen för 2026-09-11 lades till.
Loggen för 2026-09-12 lades till.
Loggen för 2026-09-13 lades till.
Loggen för 2026-09-14 lades till.
Loggen för 2026-09-15 lades till.
Loggen för 2026-09-16 lades till.
Loggen för 2026-09-17 lades till.
Loggen för 2026-09-18 lades till.
Loggen för 2026-09-19 

## log_today

Frågar efter dagens värden med `input()`, skapar ett `DailyLog`-objekt och lägger till det på användaren via `add_log`.

Datumformatet kontrolleras med `datetime.strptime`, som kastar `ValueError` om texten inte följer formatet ÅÅÅÅ-MM-DD. Det felet, `ValueError` från `DailyLog` om vikt eller kalorier är orimliga, och `ValueError` från `float()` eller `int()` om någon skriver in text där ett tal förväntas, fångas alla av samma `try/except`, eftersom de är samma feltyp.

Träningsfrågan besvaras med j eller n. Svaret görs om till små bokstäver med `.lower()` så att både J och j fungerar, och jämförs sedan mot strängen `"j"`. Resultatet av jämförelsen är redan `True` eller `False`, så det kan sparas direkt i `trained` utan en if-sats.

Går något fel sparas ingen logg, och felmeddelandet skrivs ut. Användaren får då köra `log_today(user)` igen.

In [ ]:
def log_today(user):
    """Frågar användaren om dagens värden och lägger till en DailyLog på user."""
    date_text = input("Datum (ÅÅÅÅ-MM-DD): ")

    try:
        datetime.strptime(date_text, "%Y-%m-%d")
        weight = float(input("Vikt i kg: "))
        calories = float(input("Kalorier: "))
        protein = float(input("Protein i gram: "))
        steps = int(input("Steg: "))

        trained_text = input("Tränade du idag? (j/n): ")
        trained = trained_text.lower() == "j"

        waist_text = input("Midjemått i cm (lämna tomt om du inte mätt): ")
        if waist_text == "":
            waist = None
        else:
            waist = float(waist_text)

        new_log = DailyLog(date_text, weight, calories, protein, steps, trained, waist)
        user.add_log(new_log)

    except ValueError as error:
        print(f"Loggen sparades inte. Något var fel i inmatningen: {error}")

## Test

Cellen nedan skapar en användare och lägger till fem loggar direkt i koden, så att metoderna går att testa utan att mata in allt för hand.

Den sista loggen ligger långt bak i tiden och ska därför inte räknas med i sjudagarsfönstret, men väl i sextiodagarsfönstret.

In [52]:
test_user = User("Niklas", 183, 39, "man", 1.55, 90.0, "2026-09-01")

test_user.add_log(DailyLog("2026-09-15", 90.0, 2200, 150, 8000, True))
test_user.add_log(DailyLog("2026-09-16", 89.6, 2150, 160, 9000, False))
test_user.add_log(DailyLog("2026-09-17", 89.8, 2300, 145, 7000, True))
test_user.add_log(DailyLog("2026-09-18", 89.2, 2100, 155, 8500, True))
test_user.add_log(DailyLog("2026-08-01", 95.0, 2500, 120, 5000, False))

print("Loggar totalt:", len(test_user.logs))
print("Loggar i sjudagarsfönstret:", len(test_user.get_logs(7)))
print("Medelvikt 7 dagar:", test_user.average_weight(7))
print("Medelvikt 60 dagar:", test_user.average_weight(60))
print("Viktförändring 7 dagar:", test_user.weight_change(7))
print("Träningsdagar 7 dagar:", test_user.training_days(7))
print("Träningsdagar 60 dagar:", test_user.training_days(60))

Loggen för 2026-09-15 lades till.
Loggen för 2026-09-16 lades till.
Loggen för 2026-09-17 lades till.
Loggen för 2026-09-18 lades till.
Loggen för 2026-08-01 lades till.
Loggar totalt: 5
Loggar i sjudagarsfönstret: 4
Medelvikt 7 dagar: 89.64999999999999
Medelvikt 60 dagar: 90.72
Viktförändring 7 dagar: -0.7999999999999972
Träningsdagar 7 dagar: 3
Träningsdagar 60 dagar: 3


### Test med egen inmatning

Kör cellen nedan och fyll i värden själv. Testa både ett giltigt försök och minst ett medvetet felaktigt (fel datumformat, orimlig vikt, text där ett tal förväntas), så att du sett `except`-grenen köras på riktigt.

Kör den sedan en gång till med samma datum som förra gången, för att se att `add_log` uppdaterar posten i stället för att lägga till en till.

In [53]:
log_today(test_user)

Loggen sparades inte. Något var fel i inmatningen: time data '' does not match format '%Y-%m-%d'


In [54]:
for log in test_user.logs:
    print(vars(log))

{'date': '2026-09-15', 'weight': 90.0, 'calories': 2200, 'protein': 150, 'steps': 8000, 'trained': True, 'waist': None}
{'date': '2026-09-16', 'weight': 89.6, 'calories': 2150, 'protein': 160, 'steps': 9000, 'trained': False, 'waist': None}
{'date': '2026-09-17', 'weight': 89.8, 'calories': 2300, 'protein': 145, 'steps': 7000, 'trained': True, 'waist': None}
{'date': '2026-09-18', 'weight': 89.2, 'calories': 2100, 'protein': 155, 'steps': 8500, 'trained': True, 'waist': None}
{'date': '2026-08-01', 'weight': 95.0, 'calories': 2500, 'protein': 120, 'steps': 5000, 'trained': False, 'waist': None}
